# H2-1 확장 — 6단계: 본 회귀 (가구·주차 고정효과 + Type×계층 + 클러스터 SE)

**이전 단계 요약**
- 1단계: 계층 고정 17~32주 (팀 안정구간 기준), 3계층(저/중/고) 각 834/834/832명
- 2~3단계: 캠페인 타입×주차 매핑, 가구×주차 패널 구성 (33~101주, 172,500행)
- 4단계: Type×계층 셀 크기 확인 — 저지출×TypeC=28명만 경계선, 나머지 안정적.
  선택편향 확인(저지출 25.2% vs 고지출 94.1% 도달률)
- VIF 진단: Type 활성 더미 간 VIF 1.0대, 상관계수 0.1대 — 다중공선성 문제없음
- 5단계: 평행추세 사전검정 — **저지출 계층만 깔끔히 통과(p=0.764)**,
  중지출(p=0.053)과 전체표본(p=0.087)은 경계선, 고지출(p=0.613)은 미수신 49명뿐이라 검정력 약함

**이번 단계에서 하는 일**
`spend ~ Σ(Type × tier3 상호작용 9개) + 가구고정효과 + 주차고정효과`,
표준오차는 **가구 단위 클러스터링**(84주 반복관측 패널의 자기상관 보정).

**결과 해석 시 반드시 함께 볼 것 (미리 명시)**
- 저지출 계층 결과 → 상대적으로 가장 신뢰 가능 (5단계 통과 + 셀 크기 충분)
- 중지출 계층 결과 → 캠페인 효과와 사전추세 차이가 완전히 분리되지 않은 상태
- 고지출 계층 결과 → 평행추세 검정력 부족, 신중 해석
- 저지출×TypeC 조합 → 표본 28명, 경계선(단 바닥효과 가구는 2명뿐이라 이중왜곡 위험은 낮음)


## 0. 환경 설정

In [1]:
# 최초 1회만 실행
!pip install linearmodels statsmodels --break-system-packages -q


In [2]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

DATA_DIR = "data/"   # transaction_data.csv, campaign_table.csv, campaign_desc.csv 위치

TIER_MIN_WEEK, TIER_MAX_WEEK = 17, 32   # 계층 고정 구간 (팀 안정구간 기준)
CAMP_MIN_WEEK, CAMP_MAX_WEEK = 33, 101  # 캠페인 관찰 구간
CELL_SIZE_THRESH = 30                   # 4단계에서 쓴 셀 크기 경계 기준


## 1. 파이프라인 재구성 (1~4단계와 동일)

이전 단계들과 완전히 동일한 로직. 새 내용 없음 — 6단계 입력 데이터를 만드는 과정.

In [3]:
tx = pd.read_csv(DATA_DIR + "transaction_data.csv",
                  usecols=["household_key", "DAY", "WEEK_NO", "SALES_VALUE"])
campaign_table = pd.read_csv(DATA_DIR + "campaign_table.csv")
campaign_desc  = pd.read_csv(DATA_DIR + "campaign_desc.csv")

all_households = tx["household_key"].unique()

# --- 1단계: 계층 고정 (17~32주) ---
tier_window = tx[(tx["WEEK_NO"] >= TIER_MIN_WEEK) & (tx["WEEK_NO"] <= TIER_MAX_WEEK)]
n_weeks_tier = TIER_MAX_WEEK - TIER_MIN_WEEK + 1
avg_weekly_spend = (tier_window.groupby("household_key")["SALES_VALUE"].sum() / n_weeks_tier).reindex(all_households).fillna(0)
tier3 = pd.qcut(avg_weekly_spend, 3, labels=["저지출", "중지출", "고지출"])
tier_df = pd.DataFrame({"avg_weekly_spend_pre": avg_weekly_spend, "tier3": tier3})
tier_df["was_zero_pre"] = (tier_df["avg_weekly_spend_pre"] == 0)

# --- 2단계: 캠페인 타입 x 기간 매핑 ---
day_to_week = tx[["DAY", "WEEK_NO"]].drop_duplicates().sort_values("DAY").reset_index(drop=True)
day_arr, week_arr = day_to_week["DAY"].values, day_to_week["WEEK_NO"].values
def day_to_week_lookup(day):
    idx = np.searchsorted(day_arr, day, side="right") - 1
    return week_arr[max(0, min(idx, len(day_arr) - 1))]
campaign_desc = campaign_desc.copy()
campaign_desc["START_WEEK"] = campaign_desc["START_DAY"].apply(day_to_week_lookup)
campaign_desc["END_WEEK"] = campaign_desc["END_DAY"].apply(day_to_week_lookup)
camp_full = campaign_table.merge(
    campaign_desc[["CAMPAIGN", "DESCRIPTION", "START_WEEK", "END_WEEK"]],
    on="CAMPAIGN", how="left", suffixes=("", "_desc")
)
recipients = set(campaign_table["household_key"].unique())
never_recipients = set(all_households) - recipients

# --- 3단계: 가구x주차 패널 ---
weeks_camp = list(range(CAMP_MIN_WEEK, CAMP_MAX_WEEK + 1))
panel_index = pd.MultiIndex.from_product([all_households, weeks_camp], names=["household_key", "WEEK_NO"])
panel = pd.DataFrame(index=panel_index).reset_index()
weekly_spend = (
    tx[(tx["WEEK_NO"] >= CAMP_MIN_WEEK) & (tx["WEEK_NO"] <= CAMP_MAX_WEEK)]
    .groupby(["household_key", "WEEK_NO"])["SALES_VALUE"].sum().rename("spend")
)
panel = panel.merge(weekly_spend, on=["household_key", "WEEK_NO"], how="left")
panel["spend"] = panel["spend"].fillna(0)

def build_active_weeks(camp_df, type_name):
    sub = camp_df[camp_df["DESCRIPTION_desc"] == type_name][["household_key", "START_WEEK", "END_WEEK"]].copy()
    sub["START_WEEK"] = sub["START_WEEK"].clip(lower=CAMP_MIN_WEEK)
    sub["END_WEEK"] = sub["END_WEEK"].clip(upper=CAMP_MAX_WEEK)
    sub["WEEK_NO"] = sub.apply(lambda r: list(range(int(r["START_WEEK"]), int(r["END_WEEK"]) + 1)), axis=1)
    sub = sub.explode("WEEK_NO")[["household_key", "WEEK_NO"]].drop_duplicates()
    sub["WEEK_NO"] = sub["WEEK_NO"].astype(int)
    sub[f"active_{type_name}"] = 1
    return sub

for t in ["TypeA", "TypeB", "TypeC"]:
    active = build_active_weeks(camp_full, t)
    panel = panel.merge(active, on=["household_key", "WEEK_NO"], how="left")
    panel[f"active_{t}"] = panel[f"active_{t}"].fillna(0).astype(int)

panel = panel.merge(tier_df[["tier3", "was_zero_pre"]], left_on="household_key", right_index=True, how="left")

print(f"패널 shape: {panel.shape}")
print(f"가구 수: {panel['household_key'].nunique():,} / 주차 수: {panel['WEEK_NO'].nunique()}")


패널 shape: (172500, 8)
가구 수: 2,500 / 주차 수: 69


## 2. [재확인] 4단계 셀 크기표 — 회귀 결과 해석 시 옆에 두고 볼 것

In [4]:
def cell_size_table(tier_col):
    out = {}
    for t in ["TypeA", "TypeB", "TypeC"]:
        hh_active = panel.loc[panel[f"active_{t}"] == 1, ["household_key", tier_col]].drop_duplicates()
        out[t] = hh_active.groupby(tier_col, observed=True)["household_key"].nunique()
    return pd.DataFrame(out)

cell3 = cell_size_table("tier3")
print("[Type x 3계층 수신 가구 수 -- 회귀계수 옆에서 항상 같이 볼 것]")
print(cell3)
print(f"\n{CELL_SIZE_THRESH}명 미만 조합: 결과를 참고용으로만 취급")


[Type x 3계층 수신 가구 수 -- 회귀계수 옆에서 항상 같이 볼 것]
       TypeA  TypeB  TypeC
tier3                     
저지출      187     90     28
중지출      554    305     74
고지출      772    628    295

30명 미만 조합: 결과를 참고용으로만 취급


## 3. Type × 계층 상호작용 더미 생성 (9개)

각 더미는 "이 가구가 이 계층에 속하고, 이 주에 이 타입 캠페인이 활성이었다"를 나타냄.
기준(omitted) 상태는 "그 주에 어떤 타입도 활성이 아님" — 계층 자체의 주효과는 가구고정효과에
흡수되므로(계층은 시간불변 특성) 별도로 넣지 않음.

In [5]:
for t in ["TypeA", "TypeB", "TypeC"]:
    for tier in ["저지출", "중지출", "고지출"]:
        col = f"{t}_x_{tier}"
        panel[col] = ((panel[f"active_{t}"] == 1) & (panel["tier3"] == tier)).astype(int)

interaction_cols = [f"{t}_x_{tier}" for t in ["TypeA", "TypeB", "TypeC"] for tier in ["저지출", "중지출", "고지출"]]
print("생성된 상호작용 변수:", interaction_cols)
print(panel[interaction_cols].sum().rename("가구x주차 활성 건수"))


생성된 상호작용 변수: ['TypeA_x_저지출', 'TypeA_x_중지출', 'TypeA_x_고지출', 'TypeB_x_저지출', 'TypeB_x_중지출', 'TypeB_x_고지출', 'TypeC_x_저지출', 'TypeC_x_중지출', 'TypeC_x_고지출']
TypeA_x_저지출     2166
TypeA_x_중지출     9191
TypeA_x_고지출    17297
TypeB_x_저지출      787
TypeB_x_중지출     2786
TypeB_x_고지출     9687
TypeC_x_저지출      329
TypeC_x_중지출      996
TypeC_x_고지출     4463
Name: 가구x주차 활성 건수, dtype: int64


In [6]:
# ============================================================
# [강건성 체크] TypeC x 고지출을 "최초노출" vs "재노출"로 분리
# ============================================================

def add_spell_number(df, type_col="active_TypeC"):
    """household_key별로 연속된 활성 주차를 하나의 캠페인(spell)으로 묶어 순번을 매김"""
    sub = df.loc[df[type_col] == 1, ["household_key", "WEEK_NO"]].sort_values(["household_key", "WEEK_NO"]).copy()
    sub["is_new_spell"] = (
        (sub["household_key"] != sub["household_key"].shift())
        | (sub["WEEK_NO"] != sub["WEEK_NO"].shift() + 1)
    ).astype(int)
    sub["spell_no"] = sub.groupby("household_key")["is_new_spell"].cumsum()
    return sub[["household_key", "WEEK_NO", "spell_no"]]

spell_df = add_spell_number(panel, "active_TypeC")
panel = panel.merge(spell_df, on=["household_key", "WEEK_NO"], how="left")
panel["spell_no"] = panel["spell_no"].fillna(0).astype(int)  # TypeC 비활성 주는 0

panel["TypeC_x_고지출_최초"] = (
    (panel["active_TypeC"] == 1) & (panel["tier3"] == "고지출") & (panel["spell_no"] == 1)
).astype(int)
panel["TypeC_x_고지출_재노출"] = (
    (panel["active_TypeC"] == 1) & (panel["tier3"] == "고지출") & (panel["spell_no"] >= 2)
).astype(int)

print("최초노출 가구x주차:", panel["TypeC_x_고지출_최초"].sum(),
      "/ 고유가구:", panel.loc[panel["TypeC_x_고지출_최초"]==1, "household_key"].nunique())
print("재노출 가구x주차:", panel["TypeC_x_고지출_재노출"].sum(),
      "/ 고유가구:", panel.loc[panel["TypeC_x_고지출_재노출"]==1, "household_key"].nunique())

# 기존 9개 리스트에서 TypeC_x_고지출 하나만 빼고 두 개로 교체
interaction_cols_v2 = [c for c in interaction_cols if c != "TypeC_x_고지출"] + ["TypeC_x_고지출_최초", "TypeC_x_고지출_재노출"]
print("\n최종 변수 목록:", interaction_cols_v2)

최초노출 가구x주차: 2855 / 고유가구: 295
재노출 가구x주차: 1608 / 고유가구: 131

최종 변수 목록: ['TypeA_x_저지출', 'TypeA_x_중지출', 'TypeA_x_고지출', 'TypeB_x_저지출', 'TypeB_x_중지출', 'TypeB_x_고지출', 'TypeC_x_저지출', 'TypeC_x_중지출', 'TypeC_x_고지출_최초', 'TypeC_x_고지출_재노출']


## 4. 본 회귀 — PanelOLS (가구·주차 이원고정효과 + 가구 클러스터 SE)

In [7]:
reg_df = panel.copy()
reg_df["entity"] = reg_df["household_key"]
reg_df["time"] = reg_df["WEEK_NO"]
reg_df = reg_df.set_index(["entity", "time"])

model = PanelOLS(
    dependent=reg_df["spend"],
    exog=reg_df[interaction_cols],
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
result = model.fit(cov_type="clustered", cluster_entity=True)
print(result.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                  spend   R-squared:                        0.0006
Estimator:                   PanelOLS   R-squared (Between):             -0.0188
No. Observations:              172500   R-squared (Within):               0.0004
Date:                Fri, Sep 11 2026   R-squared (Overall):             -0.0106
Time:                        23:57:38   Log-likelihood                -9.015e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      11.544
Entities:                        2500   P-value                           0.0000
Avg Obs:                       69.000   Distribution:                F(9,169923)
Min Obs:                       69.000                                           
Max Obs:                       69.000   F-statistic (robust):             2.3131
                            

In [8]:
# ============================================================
# [강건성 체크] 최초노출 vs 재노출 분리 회귀
# ============================================================
reg_df2 = panel.copy()
reg_df2["entity"] = reg_df2["household_key"]
reg_df2["time"] = reg_df2["WEEK_NO"]
reg_df2 = reg_df2.set_index(["entity", "time"])

model_v2 = PanelOLS(
    dependent=reg_df2["spend"],
    exog=reg_df2[interaction_cols_v2],
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
result_v2 = model_v2.fit(cov_type="clustered", cluster_entity=True)
print(result_v2.summary)

print()
print("="*50)
print("[핵심 비교] 최초노출 vs 재노출")
print("="*50)
compare = pd.DataFrame({
    "coef": result_v2.params[["TypeC_x_고지출_최초", "TypeC_x_고지출_재노출"]],
    "se": result_v2.std_errors[["TypeC_x_고지출_최초", "TypeC_x_고지출_재노출"]],
    "p": result_v2.pvalues[["TypeC_x_고지출_최초", "TypeC_x_고지출_재노출"]],
})
print(compare)
print(f"\n(참고) 원래 통합 계수: -6.2526, p=0.0004")

                          PanelOLS Estimation Summary                           
Dep. Variable:                  spend   R-squared:                        0.0007
Estimator:                   PanelOLS   R-squared (Between):             -0.0192
No. Observations:              172500   R-squared (Within):               0.0004
Date:                Fri, Sep 11 2026   R-squared (Overall):             -0.0108
Time:                        23:57:40   Log-likelihood                -9.015e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      11.171
Entities:                        2500   P-value                           0.0000
Avg Obs:                       69.000   Distribution:               F(10,169922)
Min Obs:                       69.000                                           
Max Obs:                       69.000   F-statistic (robust):             2.1319
                            

### 4-1. 대안 — statsmodels (linearmodels 미설치 시, 소규모 검증용)

가구 수가 많으면 더미 행렬이 커서 느릴 수 있음. linearmodels가 정상 작동하면 이 셀은 건너뛸 것.

In [9]:
# import statsmodels.api as sm
#
# reg_df2 = panel.reset_index(drop=True)
# hh_dummies = pd.get_dummies(reg_df2["household_key"], prefix="hh", drop_first=True)
# week_dummies = pd.get_dummies(reg_df2["WEEK_NO"], prefix="wk", drop_first=True)
# X = pd.concat([reg_df2[interaction_cols], hh_dummies, week_dummies], axis=1)
# X = sm.add_constant(X.astype(float))
# y = reg_df2["spend"].astype(float)
# ols_result = sm.OLS(y, X).fit(cov_type="cluster", cov_kwds={"groups": reg_df2["household_key"]})
# print(ols_result.summary())


## 5. 계수 요약 — 셀 크기와 나란히 정리

다음 단계(다중비교 보정, 위약검정)로 넘어가기 전에 결과를 셀 크기·5단계 결과와 함께 정리.

In [10]:
coef_table = result.params.to_frame("coef")
coef_table["se"] = result.std_errors
coef_table["t"] = result.tstats
coef_table["p_raw"] = result.pvalues

# 변수명(TypeX_x_계층)에서 타입/계층 분리해 셀 크기표와 조인
coef_table = coef_table.reset_index().rename(columns={"index": "변수"})
coef_table[["Type", "계층"]] = coef_table["변수"].str.split("_x_", expand=True)

cell_long = cell3.stack().reset_index()
cell_long.columns = ["계층", "Type", "수신가구수"]

coef_table = coef_table.merge(cell_long, on=["Type", "계층"], how="left")
coef_table["표본주의"] = coef_table["수신가구수"] < CELL_SIZE_THRESH

pretrend_note = {"저지출": "5단계 통과(안전)", "중지출": "5단계 경계선(p=0.053)", "고지출": "5단계 검정력 부족"}
coef_table["평행추세_참고"] = coef_table["계층"].map(pretrend_note)

print(coef_table[["Type", "계층", "coef", "se", "p_raw", "수신가구수", "표본주의", "평행추세_참고"]]
      .sort_values("coef", ascending=False).to_string(index=False))


 Type  계층      coef       se    p_raw  수신가구수  표본주의          평행추세_참고
TypeC 저지출  9.675478 6.939703 0.163253     28  True       5단계 통과(안전)
TypeB 저지출  1.442018 2.335870 0.537014     90 False       5단계 통과(안전)
TypeA 저지출  1.056302 1.431923 0.460710    187 False       5단계 통과(안전)
TypeA 중지출  0.819589 0.687613 0.233289    554 False 5단계 경계선(p=0.053)
TypeB 중지출  0.417724 1.362253 0.759116    305 False 5단계 경계선(p=0.053)
TypeA 고지출 -0.162506 0.774218 0.833748    772 False       5단계 검정력 부족
TypeC 중지출 -1.219311 4.360564 0.779768     74 False 5단계 경계선(p=0.053)
TypeB 고지출 -2.429732 0.965271 0.011832    628 False       5단계 검정력 부족
TypeC 고지출 -6.252591 1.756420 0.000371    295 False       5단계 검정력 부족


## 6. 결과 해석 체크리스트 (다음 단계로 넘어가기 전 확인)

1. 어느 Type×계층 조합의 계수가 유의(p_raw < 0.05)한가?
2. 그중 `표본주의=True`(저지출×TypeC)인 게 있는가? → 있으면 참고용으로만
3. `평행추세_참고`가 "경계선"이나 "검정력 부족"인 계층에서 나온 유의한 결과는
   캠페인 효과와 사전추세를 완전히 분리하지 못한 상태임을 결론에 명시
4. **아직 다중비교 보정 전(raw p-value)**이므로, 여기서 바로 "유의하다"고 결론 내리지 말 것
   → 다음 단계(7단계, Benjamini-Hochberg 보정)에서 최종 판단

## 다음 단계
이 결과(coef_table)를 7단계(다중비교 보정)와 8단계(위약검정, 캠페인 미수신 916가구)에 그대로 사용.